# 🎯 Proposed Target Locker

Use a **GPU runtime** in Google Colab and run the cell below. This copy uses the trained proposed hybrid model and downloads its weights from the `Proposed_model` release.

**Upload → Analyze → Preview → Select Target → Click Object → Lock & Track → Result**


In [ ]:
import os, sys, time, uuid, base64, shutil, tempfile, subprocess, urllib.request, traceback, json
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "hydra-core==1.3.2", "iopath==0.1.10", "tqdm", "pillow"],
    check=True
)

import cv2
import numpy as np
import torch
from google.colab import output
from IPython.display import display, Javascript

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → T4 GPU.")

REPO = Path("/content/Target_Locker")
WORKDIR = REPO / "SAM2_streaming-main"

if not WORKDIR.exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(
        ["git","clone","--depth","1",
         "https://github.com/CptImtiaz/Target_Locker.git",
         str(REPO)],
        check=True
    )

os.chdir(WORKDIR)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

# ============================================================
# PROPOSED HYBRID MODEL
# ============================================================

PROPOSED_DIR = REPO / "proposed"
if str(PROPOSED_DIR) not in sys.path:
    sys.path.insert(0, str(PROPOSED_DIR))

from proposed_tracker import (
    ProposedTargetLocker,
    load_proposed_models,
    point_to_bbox,
)

models, tlm, tlt = load_proposed_models(device="cuda")

print("✓ Proposed Hybrid Target Locker loaded")
print("  - fine-tuned DaSiamRPN")
print("  - Kalman motion model")
print("  - lightweight multi-pathway memory")
print("  - SAM2.1 Tiny refinement")
print("  - trained confidence fusion")

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MAX_SIDE = 960

def fit_frame(frame, max_side=MAX_SIDE):
    h,w = frame.shape[:2]
    s = min(1.0, max_side/max(h,w))
    nw = max(2, int(w*s)); nh = max(2, int(h*s))
    nw -= nw % 2; nh -= nh % 2
    if (nw,nh)!=(w,h):
        frame = cv2.resize(frame,(nw,nh),interpolation=cv2.INTER_AREA)
    return frame

# ============================================================
# ONE APP UI
# ============================================================
display(Javascript(r'''
(() => {
  const old = document.getElementById("tl-app-fixed2");
  if (old) old.remove();

  const root = document.createElement("div");
  root.id = "tl-app-fixed2";
  root.innerHTML = `
  <style>
    #tl-app-fixed2{max-width:1050px;margin:14px auto;color:#e8f7ff;font-family:Inter,system-ui,Arial,sans-serif}
    #tl-app-fixed2 *{box-sizing:border-box}
    .shell{overflow:hidden;border-radius:26px;background:radial-gradient(circle at 0 0,rgba(6,182,212,.18),transparent 30%),radial-gradient(circle at 100% 0,rgba(99,102,241,.14),transparent 28%),#030712;border:1px solid rgba(34,211,238,.22);box-shadow:0 30px 80px rgba(0,0,0,.35)}
    .head{padding:22px 26px;border-bottom:1px solid rgba(34,211,238,.13);display:flex;align-items:center;justify-content:space-between;gap:15px;flex-wrap:wrap}
    .kicker{font-size:10px;letter-spacing:4px;color:#22d3ee;font-weight:800}
    .title{font-size:31px;font-weight:950;letter-spacing:-1px;color:white;margin-top:4px}
    .sub{font-size:13px;color:#64748b;margin-top:4px}
    .chip{border:1px solid #164e63;background:#07111f;color:#67e8f9;border-radius:999px;padding:8px 12px;font-size:11px;font-weight:800;letter-spacing:1px}
    .body{padding:18px}
    .preview{position:relative;overflow:hidden;border-radius:18px;background:#000;border:1px solid rgba(34,211,238,.18);min-height:480px;display:flex;align-items:center;justify-content:center}
    .preview video,.preview canvas{width:100%;height:auto;max-height:650px;display:block;background:#000}
    .empty{display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;color:#64748b;text-align:center;padding:70px 20px}
    .upload{margin-top:8px;border:1px solid #22d3ee;background:linear-gradient(135deg,#0891b2,#4f46e5);color:white;border-radius:12px;padding:11px 20px;font-weight:900;cursor:pointer}
    .overlay{display:none;position:absolute;inset:0;z-index:10;background:rgba(2,6,23,.82);backdrop-filter:blur(7px);align-items:center;justify-content:center;flex-direction:column}
    .ring{width:76px;height:76px;border-radius:50%;border:2px solid rgba(34,211,238,.15);border-top-color:#22d3ee;border-right-color:#6366f1;animation:spin 1s linear infinite}
    .otitle{margin-top:18px;font-weight:950;letter-spacing:2px;font-size:18px}
    .osub{margin-top:5px;color:#64748b;font-size:12px;text-align:center;padding:0 18px}
    .scan{display:none;pointer-events:none;position:absolute;z-index:4;left:0;right:0;height:8%;background:linear-gradient(to bottom,transparent,rgba(34,211,238,.05),rgba(34,211,238,.75),rgba(34,211,238,.05),transparent);animation:scan 2.4s linear infinite}
    .reticle{position:absolute;z-index:5;pointer-events:none;display:none;width:62px;height:62px;transform:translate(-50%,-50%)}
    .reticle:before,.reticle:after{content:"";position:absolute;background:#22d3ee;box-shadow:0 0 12px #22d3ee}
    .reticle:before{left:30px;top:0;width:2px;height:62px}.reticle:after{top:30px;left:0;height:2px;width:62px}
    .controls{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-top:13px}
    .btn{border-radius:11px;padding:10px 15px;font-weight:900;cursor:pointer;border:1px solid #334155;background:#111827;color:#cbd5e1}
    .btn.primary{border-color:#22d3ee;background:linear-gradient(135deg,#0891b2,#4f46e5);color:white}
    .btn:disabled{opacity:.35;cursor:not-allowed}
    .info{margin-left:auto;font-family:monospace;color:#64748b;font-size:11px}
    .telemetry{margin-top:13px;display:grid;grid-template-columns:repeat(4,1fr);gap:9px}
    .card{background:#07111f;border:1px solid rgba(34,211,238,.1);border-radius:12px;padding:10px 12px}
    .card span{display:block;color:#475569;font-size:9px;letter-spacing:1.6px}.card b{display:block;margin-top:4px;color:#dff7ff;font-size:13px}
    .progress{display:none;margin-top:13px;background:#07111f;border-radius:999px;height:8px;overflow:hidden}
    .progress>div{width:0;height:100%;background:linear-gradient(90deg,#22d3ee,#6366f1);transition:width .2s ease}
    @keyframes spin{to{transform:rotate(360deg)}} @keyframes scan{from{top:-8%}to{top:100%}}
    @media(max-width:760px){.telemetry{grid-template-columns:repeat(2,1fr)}.info{width:100%;margin-left:0}}
  </style>

  <div class="shell">
    <div class="head">
      <div>
        <div class="kicker">PROPOSED HYBRID // SINGLE OBJECT TRACKING</div>
        <div class="title">TARGET LOCKER</div>
        <div class="sub">Upload a video. Acquire one target. Track only that target.</div>
      </div>
      <div id="chip" class="chip">SYSTEM READY</div>
    </div>

    <div class="body">
      <input id="file" type="file" accept="video/*" style="display:none">

      <div class="preview">
        <div id="empty" class="empty">
          <div style="font-size:46px">◉</div>
          <div style="font-size:18px;font-weight:900;color:#cbd5e1">LOAD TARGET FEED</div>
          <div>Select one video file to initialize the tracker.</div>
          <button id="upload" class="upload">UPLOAD VIDEO</button>
        </div>

        <video id="video" controls playsinline style="display:none"></video>
        <canvas id="canvas" style="display:none"></canvas>
        <div id="scan" class="scan"></div>
        <div id="reticle" class="reticle"></div>

        <div id="overlay" class="overlay">
          <div class="ring"></div>
          <div id="otitle" class="otitle">ANALYZING VIDEO</div>
          <div id="osub" class="osub">Preparing stream…</div>
        </div>
      </div>

      <div class="controls">
        <button id="newVideo" class="btn" style="display:none">NEW VIDEO</button>
        <button id="select" class="btn" disabled>SELECT TARGET</button>
        <button id="track" class="btn primary" disabled>LOCK & TRACK</button>
        <div id="info" class="info">NO VIDEO LOADED</div>
      </div>

      <div id="telemetry" class="telemetry" style="display:none">
        <div class="card"><span>RESOLUTION</span><b id="res">—</b></div>
        <div class="card"><span>FRAME RATE</span><b id="fps">—</b></div>
        <div class="card"><span>FRAMES</span><b id="frames">—</b></div>
        <div class="card"><span>DURATION</span><b id="duration">—</b></div>
      </div>

      <div id="progress" class="progress"><div id="bar"></div></div>
    </div>
  </div>`;

  document.body.appendChild(root);

  const q = s => root.querySelector(s);
  window.TLAPP = {
    root,
    file:q("#file"), upload:q("#upload"), newVideo:q("#newVideo"),
    empty:q("#empty"), video:q("#video"), canvas:q("#canvas"),
    ctx:q("#canvas").getContext("2d"), scan:q("#scan"), reticle:q("#reticle"),
    overlay:q("#overlay"), otitle:q("#otitle"), osub:q("#osub"),
    select:q("#select"), track:q("#track"), chip:q("#chip"),
    info:q("#info"), telemetry:q("#telemetry"), progress:q("#progress"), bar:q("#bar"),
    res:q("#res"), fps:q("#fps"), frames:q("#frames"), duration:q("#duration"),
    blobURL:null, selectedFile:null, target:null
  };

  TLAPP.upload.onclick = () => TLAPP.file.click();
  TLAPP.newVideo.onclick = () => TLAPP.file.click();

  google.colab.output.setIframeHeight(document.body.scrollHeight,true);
})();
'''))

# ============================================================
# Upload
# ============================================================
file_meta = output.eval_js(r'''
(async () => {
  const A = window.TLAPP;

  const f = await new Promise(resolve => {
    A.file.onchange = () => resolve(A.file.files?.[0] || null);
  });

  if(!f) return null;
  A.selectedFile = f;

  if(A.blobURL) URL.revokeObjectURL(A.blobURL);
  A.blobURL = URL.createObjectURL(f);

  A.empty.style.display = "none";
  A.canvas.style.display = "none";
  A.video.style.display = "block";
  A.video.src = A.blobURL;
  A.video.load();

  A.newVideo.style.display = "inline-block";
  A.select.disabled = true;
  A.track.disabled = true;
  A.telemetry.style.display = "none";

  A.overlay.style.display = "flex";
  A.otitle.textContent = "ANALYZING VIDEO";
  A.osub.textContent = "Uploading stream to Colab…";
  A.chip.textContent = "ANALYZING";
  A.info.textContent = f.name.toUpperCase();

  A.progress.style.display = "block";
  A.bar.style.width = "2%";

  return {name:f.name,size:f.size,type:f.type || "video/mp4"};
})()
''')

if not file_meta:
    raise RuntimeError("No video selected.")

name = file_meta["name"]
size = int(file_meta["size"])
suffix = Path(name).suffix or ".mp4"
video_path = Path("/content") / f"target-locker-{uuid.uuid4().hex}{suffix}"

CHUNK = 512 * 1024
total_chunks = (size + CHUNK - 1) // CHUNK

with open(video_path, "wb") as f:
    for i in range(total_chunks):
        start = i * CHUNK
        end = min(size, start + CHUNK)

        chunk_b64 = output.eval_js(f'''
        (async () => {{
          const A = window.TLAPP;
          const buf = await A.selectedFile.slice({start},{end}).arrayBuffer();
          const bytes = new Uint8Array(buf);
          let binary = "";
          const step = 0x8000;
          for(let j=0;j<bytes.length;j+=step){{
            binary += String.fromCharCode(...bytes.subarray(j,j+step));
          }}
          A.bar.style.width = "{5 + int(((i+1)/max(total_chunks,1))*60)}%";
          A.osub.textContent = "Uploading video stream… {round(((i+1)/max(total_chunks,1))*100)}%";
          return btoa(binary);
        }})()
        ''')

        f.write(base64.b64decode(chunk_b64))

# ============================================================
# Analyze
# ============================================================
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError("Could not open uploaded video.")

fps = float(cap.get(cv2.CAP_PROP_FPS))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, first_frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not decode uploaded video.")
if not np.isfinite(fps) or fps <= 0:
    fps = 25.0

duration = frames / fps if frames > 0 else 0.0
first_frame = fit_frame(first_frame)

output.eval_js(f'''
(() => {{
  const A = window.TLAPP;
  A.bar.style.width = "100%";
  A.otitle.textContent = "ANALYSIS COMPLETE";
  A.osub.textContent = "Video verified. Preview starting…";

  A.res.textContent = {json.dumps(f"{width} × {height}")};
  A.fps.textContent = {json.dumps(f"{fps:.2f} FPS")};
  A.frames.textContent = {json.dumps(str(frames))};
  A.duration.textContent = {json.dumps(f"{duration:.1f}s")};
  A.telemetry.style.display = "grid";

  setTimeout(() => {{
    A.overlay.style.display = "none";
    A.progress.style.display = "none";
    A.chip.textContent = "VIDEO READY";
    A.info.textContent = "PREVIEW ACTIVE // SELECT TARGET WHEN READY";
    A.select.disabled = false;
    A.video.currentTime = 0;
    A.video.play().catch(()=>{{}});
  }}, 350);

  return true;
}})()
''')

# ============================================================
# SINGLE CONTINUOUS INTERACTION BLOCK:
# SELECT TARGET -> click object -> LOCK & TRACK
# ============================================================
selection = output.eval_js(r'''
(async () => {
  const A = window.TLAPP;

  // Wait for the Select Target button.
  await new Promise(resolve => {
    A.select.onclick = () => resolve();
  });

  A.video.pause();

  // Force frame 1.
  try {
    A.video.currentTime = 0;
  } catch(e) {}

  await new Promise(resolve => {
    let finished = false;

    const done = () => {
      if(finished) return;
      finished = true;
      resolve();
    };

    A.video.addEventListener("seeked", done, {once:true});
    A.video.addEventListener("loadeddata", done, {once:true});

    // Browser safety fallback.
    setTimeout(done, 700);
  });

  const w = A.video.videoWidth;
  const h = A.video.videoHeight;

  if(!w || !h){
    throw new Error("Video frame is not ready.");
  }

  A.canvas.width = w;
  A.canvas.height = h;
  A.ctx.drawImage(A.video,0,0,w,h);

  A.video.style.display = "none";
  A.canvas.style.display = "block";

  A.scan.style.display = "block";
  A.reticle.style.display = "none";

  A.select.textContent = "CLICK OBJECT";
  A.select.disabled = true;

  A.chip.textContent = "TARGET ACQUISITION";
  A.info.textContent = "CLICK THE TARGET DIRECTLY ON FRAME 1";

  // Wait for target click.
  const click = await new Promise(resolve => {
    A.canvas.onclick = ev => {
      const r = A.canvas.getBoundingClientRect();

      const x = (ev.clientX-r.left) * A.canvas.width / r.width;
      const y = (ev.clientY-r.top) * A.canvas.height / r.height;

      A.target = [x,y];

      A.reticle.style.left = ((x/A.canvas.width)*100) + "%";
      A.reticle.style.top = ((y/A.canvas.height)*100) + "%";
      A.reticle.style.display = "block";
      A.scan.style.display = "none";

      A.canvas.onclick = null;

      A.track.disabled = false;
      A.select.textContent = "TARGET SELECTED";

      A.chip.textContent = "TARGET LOCKED";
      A.info.textContent = "TARGET ACQUIRED // PRESS LOCK & TRACK";

      resolve({
        x:x,
        y:y,
        cw:A.canvas.width,
        ch:A.canvas.height
      });
    };
  });

  // Wait for Lock & Track.
  await new Promise(resolve => {
    A.track.onclick = () => resolve();
  });

  A.track.disabled = true;
  A.select.disabled = true;

  A.overlay.style.display = "flex";
  A.otitle.textContent = "TRACKING TARGET";
  A.osub.textContent = "Proposed Hybrid is tracking with Siamese, Kalman, memory, SAM2.1 and learned fusion…";

  A.chip.textContent = "TRACKING";
  A.info.textContent = "TARGET PURSUIT ACTIVE";

  A.progress.style.display = "block";
  A.bar.style.width = "18%";

  return click;
})()
''')

x = int(float(selection["x"]) * first_frame.shape[1] / max(float(selection["cw"]),1))
y = int(float(selection["y"]) * first_frame.shape[0] / max(float(selection["ch"]),1))

x = int(np.clip(x,0,first_frame.shape[1]-1))
y = int(np.clip(y,0,first_frame.shape[0]-1))
target_point = [x,y]

# Convert the mouse click to an initialization bbox using SAM2.1 Tiny
# inside the proposed model stack.
init_bbox = point_to_bbox(
    first_frame,
    target_point,
    models,
)

print("✓ Initial target bbox:", init_bbox)

# ============================================================
# Track — PROPOSED HYBRID
# ============================================================
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError("Could not reopen video.")

ok, frame = cap.read()
if not ok:
    cap.release()
    raise RuntimeError("Could not read first frame.")

frame = fit_frame(frame)
h,w = frame.shape[:2]

job_dir = Path(tempfile.mkdtemp(prefix="proposed-target-locker-", dir="/content"))
raw_path = job_dir / "tracked_raw.mp4"
final_path = job_dir / "target_locked.mp4"

writer = cv2.VideoWriter(
    str(raw_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w,h)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

tracker = ProposedTargetLocker(
    models,
    tlm,
    tlt,
    fusion_threshold=0.30,
    sam2_refine_threshold=0.30,
    top_k=3,
)

tracker.initialize(frame, init_bbox)

processed = 0
last_ui = -1
idx = 0

try:
    with torch.inference_mode():
        while True:
            if idx > 0:
                ok, frame = cap.read()
                if not ok:
                    break

                frame = fit_frame(frame)
                bbox = tracker.update(frame)
            else:
                bbox = init_bbox

            result = frame.copy()

            if bbox is not None:
                bx,by,bw,bh = [int(round(v)) for v in bbox]

                bx = max(0, min(bx, result.shape[1]-1))
                by = max(0, min(by, result.shape[0]-1))
                bw = max(2, min(bw, result.shape[1]-bx))
                bh = max(2, min(bh, result.shape[0]-by))

                x2 = bx + bw
                y2 = by + bh

                c = (255,255,0)
                cv2.rectangle(result,(bx,by),(x2,y2),c,2)

                cv2.putText(
                    result,"PROPOSED TARGET LOCK",
                    (bx,max(27,by-9)),
                    cv2.FONT_HERSHEY_DUPLEX,
                    0.6,c,2,cv2.LINE_AA
                )

                if idx > 0 and tracker.last_signals is not None:
                    fused = tracker.last_signals.get("fusion", 0.0)
                    cv2.putText(
                        result,
                        f"Fusion: {fused:.2f}",
                        (bx, min(result.shape[0]-12, y2+24)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,c,1,cv2.LINE_AA
                    )

            writer.write(result)

            idx += 1
            processed += 1

            if frames > 0:
                pct = int((processed/frames)*100)
                bucket = pct // 5

                if bucket != last_ui:
                    last_ui = bucket
                    bar_pct = min(92, 18 + int((processed/frames)*74))

                    output.eval_js(f"""
                    (() => {{
                      const A = window.TLAPP;
                      A.bar.style.width = "{bar_pct}%";
                      A.osub.textContent = "Proposed model tracking frame {processed} / {frames}";
                      return true;
                    }})()
                    """)

finally:
    cap.release()
    writer.release()
    torch.cuda.empty_cache()

subprocess.run(
    [
        "ffmpeg","-nostdin","-y",
        "-i",str(raw_path),
        "-an",
        "-c:v","libx264",
        "-preset","fast",
        "-crf","22",
        "-pix_fmt","yuv420p",
        "-movflags","+faststart",
        str(final_path)
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# ============================================================
# Result in SAME preview
# ============================================================
if final_path.stat().st_size <= 100*1024*1024:
    data_url = "data:video/mp4;base64," + base64.b64encode(final_path.read_bytes()).decode("ascii")

    output.eval_js(f'''
    (() => {{
      const A = window.TLAPP;

      A.canvas.style.display = "none";
      A.reticle.style.display = "none";
      A.scan.style.display = "none";

      A.video.style.display = "block";
      A.video.src = {json.dumps(data_url)};
      A.video.load();

      A.overlay.style.display = "none";
      A.progress.style.display = "none";

      A.chip.textContent = "TRACK COMPLETE";
      A.info.textContent = "TARGET-LOCKED RESULT // {processed} FRAMES";

      A.video.play().catch(()=>{{}});

      return true;
    }})()
    ''')
else:
    output.eval_js(f'''
    (() => {{
      const A = window.TLAPP;

      A.overlay.style.display = "none";
      A.progress.style.display = "none";

      A.chip.textContent = "TRACK COMPLETE";
      A.info.textContent = "RESULT READY // {processed} FRAMES // TOO LARGE FOR INLINE PREVIEW";

      return true;
    }})()
    ''')

print(f"✅ Tracking complete: {processed} frames")
print(f"Output: {final_path}")
